# Project : Impact of Sample Processing (Fresh vs. Fixed vs. Fixed-Stored) on Single-Cell Transcriptomic Profiling of Diseased PBMCs in Primary Myelofibrosis

### Section 1: Import Library and Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [2]:
!pip install 'scanpy[leiden]'

In [3]:
import scanpy as sc
import anndata as ad 

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
path = '/content/drive/MyDrive/PBMC/'

In [12]:
# Set path to your PBMC files in Google Drive
path = '/content/drive/MyDrive/PBMC/'  # ← Update this to your actual path

# Load all three datasets
adata_fresh = sc.read_10x_h5(path + '10k_3p_Human_diseased_PBMC_Myelofibrosis_Fresh_count_filtered_feature_bc_matrix.h5')
adata_fix = sc.read_10x_h5(path + '10k_3p_Human_diseased_PBMC_Myelofibrosis_Fix_count_filtered_feature_bc_matrix.h5')
adata_stored = sc.read_10x_h5(path + '10k_3p_Human_diseased_PBMC_Myelofibrosis_Fix_stored_count_filtered_feature_bc_matrix.h5')

# Fix duplicate gene names immediately
adata_fresh.var_names_make_unique()
adata_fix.var_names_make_unique()
adata_stored.var_names_make_unique()

print("✅ Loaded successfully")
print(f"  Fresh:        {adata_fresh.shape}")
print(f"  Fixed:        {adata_fix.shape}")
print(f"  Fixed_Stored: {adata_stored.shape}")


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


✅ Loaded successfully
  Fresh:        (12516, 38606)
  Fixed:        (11538, 38606)
  Fixed_Stored: (5825, 38606)


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [13]:
# Label each dataset with its condition
adata_fresh.obs['condition'] = 'Fresh'
adata_fix.obs['condition'] = 'Fixed'
adata_stored.obs['condition'] = 'Fixed_Stored'

In [14]:
# Shape check
print("=== Dataset Shapes ===")
print(f"Fresh:        {adata_fresh.shape[0]:,} cells × {adata_fresh.shape[1]:,} genes")
print(f"Fixed:        {adata_fix.shape[0]:,} cells × {adata_fix.shape[1]:,} genes")
print(f"Fixed_Stored: {adata_stored.shape[0]:,} cells × {adata_stored.shape[1]:,} genes")

# Gene names match?
print(f"\nGenes match (Fresh vs Fixed): {(adata_fresh.var_names == adata_fix.var_names).all()}")
print(f"Genes match (Fresh vs Stored): {(adata_fresh.var_names == adata_stored.var_names).all()}")

# Preview obs
print("\n=== .obs columns ===")
print(list(adata_fresh.obs.columns))
display(adata_fresh.obs.head(3))

# Preview var
print("\n=== .var (first 5 genes) ===")
display(adata_fresh.var.head())


=== Dataset Shapes ===
Fresh:        12,516 cells × 38,606 genes
Fixed:        11,538 cells × 38,606 genes
Fixed_Stored: 5,825 cells × 38,606 genes

Genes match (Fresh vs Fixed): True
Genes match (Fresh vs Stored): True

=== .obs columns ===
['condition']


,condition
AAACCAAAGGAATGGT-1,Fresh
AAACCAAAGGGCAGGT-1,Fresh
AAACCAAAGGTTGCCG-1,Fresh



=== .var (first 5 genes) ===


,gene_ids,feature_types,genome
DDX11L2,ENSG00000290825,Gene Expression,GRCh38
MIR1302-2HG,ENSG00000243485,Gene Expression,GRCh38
FAM138A,ENSG00000237613,Gene Expression,GRCh38
ENSG00000290826,ENSG00000290826,Gene Expression,GRCh38
OR4F5,ENSG00000186092,Gene Expression,GRCh38


In [15]:
# Make all names unique before merging
adata_fresh.obs_names_make_unique()
adata_fix.obs_names_make_unique()
adata_stored.obs_names_make_unique()

# Concatenate
adata = ad.concat(
    [adata_fresh, adata_fix, adata_stored],
    label='condition',
    keys=['Fresh', 'Fixed', 'Fixed_Stored'],
    join='inner',
    index_unique='-'
)

print(f"✅ Merged: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
print(f"\nCells per condition:")
print(adata.obs['condition'].value_counts())


✅ Merged: 29,879 cells × 38,606 genes

Cells per condition:
condition
Fresh           12516
Fixed           11538
Fixed_Stored     5825
Name: count, dtype: int64


### Section 2: Quality Control